<a href="https://colab.research.google.com/github/hailleenvarela/desafio-motriz-dados/blob/main/02_carga_modelagem_bigquery_motriz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 — Carga e modelagem no BigQuery | Case Motriz

**Objetivo:** carregar as extrações e os dados tratados do Notebook 01 para o BigQuery Sandbox, criar três camadas (`bronze`, `silver`, `gold`) e materializar os três indicadores em BigQuery Standard SQL.

**Pré-requisitos**
1. Execute o Notebook 01 até o final sem erros. Ele deve gerar `bronze/`, `silver/frequencia_todas_redes.csv` e os arquivos em `resultados/` no Google Drive.
2. Use o mesmo Google Drive/conta do Colab e selecione o projeto GCP que você já criou.
3. No Google Cloud, confirme que o BigQuery API está habilitado no projeto. O Sandbox não exige cartão de crédito.

**Segurança e reexecução:** a autenticação usa sua sessão Google no Colab; não cole chaves de conta de serviço. As tabelas são recriadas com `WRITE_TRUNCATE`/`CREATE OR REPLACE`, portanto executar novamente substitui os dados da mesma tabela em vez de duplicá-los. Não compartilhe o projeto nem os dados pessoais além do necessário.


In [2]:
%pip -q install --upgrade google-cloud-bigquery pyarrow db-dtypes

In [1]:
from pathlib import Path
import re, json, os
import pandas as pd
from google.colab import drive, auth
from google.cloud import bigquery
import google.auth

# Monta o mesmo Drive usado no Notebook 01.
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/motriz_case')
BRONZE = BASE / 'bronze'
SILVER = BASE / 'silver'
RESULTADOS = BASE / 'resultados'

for p in [BRONZE, SILVER, RESULTADOS]:
    if not p.exists():
        raise FileNotFoundError(f'Pasta não encontrada: {p}. Execute o Notebook 01 até o fim primeiro.')

print('Pastas encontradas:')
for p in [BRONZE, SILVER, RESULTADOS]:
    print(f'- {p}: {len(list(p.iterdir()))} arquivo(s)')


Mounted at /content/drive
Pastas encontradas:
- /content/drive/MyDrive/motriz_case/bronze: 6 arquivo(s)
- /content/drive/MyDrive/motriz_case/silver: 4 arquivo(s)
- /content/drive/MyDrive/motriz_case/resultados: 9 arquivo(s)


## 1. Configurar o projeto e autenticar

Informe o **ID do projeto** (não é o nome de exibição). Você encontra esse valor no seletor de projeto do Google Cloud. O dataset location precisa ser igual para os três datasets. `US` é um padrão comum, mas se você já criou datasets em outra região, ajuste aqui para a região existente.

In [6]:
PROJECT_ID = input('Digite o ID do projeto Google Cloud: ').strip()
BQ_LOCATION = "southamerica-east1"
DATASET_BRONZE = 'motriz_bronze'
DATASET_SILVER = 'motriz_silver'
DATASET_GOLD = 'motriz_gold'

if not PROJECT_ID or not re.fullmatch(r'[a-z][a-z0-9-]{4,61}[a-z0-9]', PROJECT_ID):
    raise ValueError('ID de projeto inválido. Copie o Project ID do Google Cloud Console.')

auth.authenticate_user()
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location=BQ_LOCATION)
print('Cliente BigQuery configurado para o projeto:', client.project)


Digite o ID do projeto Google Cloud: motriz-desafio-dados
Cliente BigQuery configurado para o projeto: motriz-desafio-dados


## 2. Criar os datasets das três camadas

O código cria os datasets se ainda não existirem. Se já existirem, confere a região; não apaga os datasets nem outras tabelas.

In [7]:
def ensure_dataset(dataset_name):
    dataset_id = f'{PROJECT_ID}.{dataset_name}'
    try:
        ds = client.get_dataset(dataset_id)
        if ds.location.upper() != BQ_LOCATION.upper():
            raise ValueError(
                f'O dataset {dataset_id} existe na região {ds.location}, mas BQ_LOCATION está {BQ_LOCATION}. '
                'Ajuste BQ_LOCATION para a região do dataset.'
            )
        print('Dataset já existe:', dataset_id, '| região:', ds.location)
        return ds
    except Exception as exc:
        # Só tenta criar quando o dataset não existe. Erros de permissão/região não devem ser mascarados.
        if isinstance(exc, ValueError):
            raise
        try:
            ds = bigquery.Dataset(dataset_id)
            ds.location = BQ_LOCATION
            ds = client.create_dataset(ds, exists_ok=True)
            print('Dataset criado:', dataset_id, '| região:', ds.location)
            return ds
        except Exception:
            raise

for dataset_name in [DATASET_BRONZE, DATASET_SILVER, DATASET_GOLD]:
    ensure_dataset(dataset_name)


Dataset criado: motriz-desafio-dados.motriz_bronze | região: southamerica-east1
Dataset criado: motriz-desafio-dados.motriz_silver | região: southamerica-east1
Dataset criado: motriz-desafio-dados.motriz_gold | região: southamerica-east1


## 3. Funções de carga idempotente

- Os CSVs de extração e os 12 arquivos originais da Rede C vão para Bronze.
- A tabela consolidada `frequencia_todas_redes` vai para Silver com tipos explícitos, particionamento por data e clusterização por rede/escola/turma.
- Arquivos de auditoria de qualidade vão para Silver.
- Gold será gerada por SQL a partir da Silver, para demonstrar que o BigQuery produz os indicadores.


In [8]:
def safe_table_name(name):
    name = Path(name).stem.lower()
    name = re.sub(r'[^a-z0-9_]', '_', name)
    if not name or not name[0].isalpha() and name[0] != '_':
        name = 't_' + name
    return name[:1024]

def load_csv_table(path, dataset_name, table_name=None, schema=None, autodetect=True,
                   partition_field=None, clustering_fields=None):
    path = Path(path)
    table_name = table_name or safe_table_name(path.name)
    table_id = f'{PROJECT_ID}.{dataset_name}.{table_name}'
    config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=autodetect,
        schema=schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        allow_quoted_newlines=True,
        encoding='UTF-8',
    )
    if partition_field:
        config.time_partitioning = bigquery.TimePartitioning(
            type_=bigquery.TimePartitioningType.DAY,
            field=partition_field,
        )
    if clustering_fields:
        config.clustering_fields = clustering_fields
    with open(path, 'rb') as f:
        job = client.load_table_from_file(f, table_id, job_config=config, rewind=True)
    job.result()
    table = client.get_table(table_id)
    print(f'{table_id}: {table.num_rows:,} linhas carregadas')
    return table_id


## 4. Carregar Bronze

Cada CSV extraído do dump relacional ou expandido do NDJSON vira uma tabela. Os 12 CSVs originais da Rede C também são mantidos como tabelas independentes, preservando diferenças entre arquivos. O autodetect é usado apenas nessa camada de exploração; a Silver terá esquema explícito.

In [9]:
# Tabelas estruturadas geradas pelo Notebook 01
bronze_csvs = sorted(BRONZE.glob('*.csv'))
if not bronze_csvs:
    raise FileNotFoundError(f'Nenhum CSV de Bronze encontrado em {BRONZE}. Execute o Notebook 01.')
bronze_loaded = []
for path in bronze_csvs:
    bronze_loaded.append(load_csv_table(path, DATASET_BRONZE))

# CSVs originais da Rede C, um por arquivo. Como há separadores/codificações diferentes,
# lemos cada arquivo de forma flexível e o serializamos em UTF-8/comma antes da carga.
import io
rede_c_dir = BASE / 'dados' / 'rede_c'
rede_c_files = sorted(rede_c_dir.glob('*.csv')) if rede_c_dir.exists() else []
if rede_c_files:
    print(f'Carregando {len(rede_c_files)} CSV(s) original(is) da Rede C em Bronze...')
    for path in rede_c_files:
        raw_df = None
        read_errors = []
        for encoding in ('utf-8-sig', 'utf-8', 'cp1252', 'latin-1'):
            try:
                raw_df = pd.read_csv(path, sep=None, engine='python', dtype=str, encoding=encoding)
                break
            except Exception as exc:
                read_errors.append(f'{encoding}: {exc}')
        if raw_df is None:
            raise ValueError(f'Falha ao ler CSV original {path.name}: {read_errors[-2:]}')
        # Padroniza apenas transporte (UTF-8 e separador); os nomes/valores originais das colunas são mantidos.
        payload = raw_df.to_csv(index=False, encoding='utf-8').encode('utf-8')
        table_id = f'{PROJECT_ID}.{DATASET_BRONZE}.rede_c_raw_{safe_table_name(path.name)}'
        config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            allow_quoted_newlines=True,
            encoding='UTF-8',
        )
        job = client.load_table_from_file(io.BytesIO(payload), table_id, job_config=config, rewind=True)
        job.result()
        print(f'{table_id}: {client.get_table(table_id).num_rows:,} linhas carregadas')
else:
    print('Aviso: pasta com CSVs originais da Rede C não encontrada; as extrações Bronze continuam sendo carregadas.')

print('Tabelas Bronze carregadas:', len(bronze_loaded))


motriz-desafio-dados.motriz_bronze.rede_a_aluno: 417 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_a_escola: 5 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_a_frequencia: 26,177 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_a_matricula: 417 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_a_turma: 17 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_b_frequencia_extraida: 19,680 linhas carregadas
Carregando 12 CSV(s) original(is) da Rede C em Bronze...
motriz-desafio-dados.motriz_bronze.rede_c_raw_frequencia_5119_agosto_2026: 1,808 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_c_raw_frequencia_5119_outubro_2026: 1,809 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_c_raw_frequencia_5119_setembro_2026: 1,810 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_c_raw_frequencia_5906_agosto_2026: 1,733 linhas carregadas
motriz-desafio-dados.motriz_bronze.rede_c_raw_frequencia_5906_outubro_2026: 1,705 linhas ca

## 5. Carregar Silver com esquema explícito

A tabela principal tem uma linha por registro de frequência válido após aplicação das regras de negócio. `data` é usada para particionamento diário; os campos de rede/escola/turma ajudam a filtrar consultas comuns.

In [10]:
silver_path = SILVER / 'frequencia_todas_redes.csv'
if not silver_path.exists():
    # Alternativa: montar a tabela consolidada a partir dos três arquivos por rede.
    per_network = sorted(SILVER.glob('frequencia_rede_*.csv'))
    if not per_network:
        raise FileNotFoundError('Não encontrei frequencia_todas_redes.csv nem frequencia_rede_*.csv. Execute o Notebook 01 até o final.')
    frames = [pd.read_csv(p, dtype=str) for p in per_network]
    pd.concat(frames, ignore_index=True).to_csv(silver_path, index=False, encoding='utf-8')

silver_schema = [
    bigquery.SchemaField('rede', 'STRING'),
    bigquery.SchemaField('codigo_inep', 'STRING'),
    bigquery.SchemaField('escola', 'STRING'),
    bigquery.SchemaField('turma', 'STRING'),
    bigquery.SchemaField('turno', 'STRING'),
    bigquery.SchemaField('identificador_origem', 'STRING'),
    bigquery.SchemaField('identificador_aluno', 'STRING'),
    bigquery.SchemaField('nome_aluno', 'STRING'),
    bigquery.SchemaField('data_nascimento', 'DATE'),
    bigquery.SchemaField('data', 'DATE'),
    bigquery.SchemaField('mes', 'STRING'),
    bigquery.SchemaField('status', 'STRING'),
    bigquery.SchemaField('presencas', 'INT64'),
    bigquery.SchemaField('inicio_matricula', 'DATE'),
    bigquery.SchemaField('fim_matricula', 'DATE'),
]
load_csv_table(
    silver_path, DATASET_SILVER, table_name='frequencia_todas_redes',
    schema=silver_schema, autodetect=False,
    partition_field='data', clustering_fields=['rede', 'codigo_inep', 'turma']
)

# Tabelas de auditoria, quando disponíveis.
for filename, table_name in [
    ('resumo_qualidade.csv', 'resumo_qualidade'),
    ('motivos_descarte.csv', 'motivos_descarte'),
]:
    path = RESULTADOS / filename
    if path.exists() and path.stat().st_size > 0:
        load_csv_table(path, DATASET_SILVER, table_name=table_name, autodetect=True)


motriz-desafio-dados.motriz_silver.frequencia_todas_redes: 40,364 linhas carregadas
motriz-desafio-dados.motriz_silver.resumo_qualidade: 3 linhas carregadas
motriz-desafio-dados.motriz_silver.motivos_descarte: 17,246 linhas carregadas


## 6. Criar as tabelas analíticas Gold via BigQuery Standard SQL

As consultas abaixo recalculam os resultados no BigQuery, somando numeradores e denominadores — não tirando média de percentuais. `CREATE OR REPLACE TABLE` permite reexecução sem duplicar dados.

In [11]:
silver_table = f'`{PROJECT_ID}.{DATASET_SILVER}.frequencia_todas_redes`'
gold_escola = f'{PROJECT_ID}.{DATASET_GOLD}.frequencia_escola_mes'
gold_turma = f'{PROJECT_ID}.{DATASET_GOLD}.frequencia_turma_mes'
gold_alunos = f'{PROJECT_ID}.{DATASET_GOLD}.alunos_abaixo_75'

sql_escola = f'''CREATE OR REPLACE TABLE `{gold_escola}` AS
SELECT
  rede,
  codigo_inep,
  mes,
  COUNT(*) AS registros_validos,
  SUM(presencas) AS presencas,
  ROUND(SAFE_DIVIDE(SUM(presencas), COUNT(*)) * 100, 2) AS frequencia_pct
FROM {silver_table}
GROUP BY rede, codigo_inep, mes'''

sql_turma = f'''CREATE OR REPLACE TABLE `{gold_turma}` AS
SELECT
  rede,
  codigo_inep,
  turma,
  mes,
  COUNT(*) AS registros_validos,
  SUM(presencas) AS presencas,
  ROUND(SAFE_DIVIDE(SUM(presencas), COUNT(*)) * 100, 2) AS frequencia_pct
FROM {silver_table}
GROUP BY rede, codigo_inep, turma, mes'''

sql_alunos = f'''CREATE OR REPLACE TABLE `{gold_alunos}` AS
WITH frequencia_aluno AS (
  SELECT
    rede,
    identificador_aluno,
    ARRAY_AGG(nome_aluno IGNORE NULLS ORDER BY data LIMIT 1)[SAFE_OFFSET(0)] AS nome_aluno,
    ARRAY_AGG(data_nascimento IGNORE NULLS ORDER BY data LIMIT 1)[SAFE_OFFSET(0)] AS data_nascimento,
    COUNT(*) AS registros_validos,
    SUM(presencas) AS presencas,
    ROUND(SAFE_DIVIDE(SUM(presencas), COUNT(*)) * 100, 2) AS frequencia_pct
  FROM {silver_table}
  GROUP BY rede, identificador_aluno
)
SELECT
  rede, identificador_aluno, nome_aluno, data_nascimento,
  registros_validos, presencas, frequencia_pct
FROM frequencia_aluno
WHERE frequencia_pct < 75.00'''

for sql in [sql_escola, sql_turma, sql_alunos]:
    client.query(sql).result()
    print('Consulta executada com sucesso.')


Consulta executada com sucesso.
Consulta executada com sucesso.
Consulta executada com sucesso.


## 7. Conferir tabelas e amostras

Confira as contagens e visualize uma amostra. Compare os resultados com os CSVs do Notebook 01 antes de considerar a entrega concluída.

In [ ]:
for table_id in [
    f'{PROJECT_ID}.{DATASET_SILVER}.frequencia_todas_redes',
    gold_escola, gold_turma, gold_alunos,
]:
    table = client.get_table(table_id)
    print(f'{table_id}: {table.num_rows:,} linhas; {len(table.schema)} colunas')

for table_id in [gold_escola, gold_turma, gold_alunos]:
    print('\nAmostra:', table_id)
    df_preview = client.query(f'SELECT * FROM `{table_id}` LIMIT 10').to_dataframe()
    display(df_preview)


## 8. Exportar os resultados do BigQuery para CSV

Esta etapa baixa os resultados analíticos calculados no BigQuery para a pasta `resultados/`, mantendo os nomes e a ordem das colunas exigidas pelo enunciado. O BigQuery permanece como fonte das tabelas Gold.

In [ ]:
exports = {
    'frequencia_escola_mes.csv': gold_escola,
    'frequencia_turma_mes.csv': gold_turma,
    'alunos_abaixo_75.csv': gold_alunos,
}
expected_columns = {
    'frequencia_escola_mes.csv': ['rede','codigo_inep','mes','registros_validos','presencas','frequencia_pct'],
    'frequencia_turma_mes.csv': ['rede','codigo_inep','turma','mes','registros_validos','presencas','frequencia_pct'],
    'alunos_abaixo_75.csv': ['rede','identificador_aluno','nome_aluno','data_nascimento','registros_validos','presencas','frequencia_pct'],
}
RESULTADOS.mkdir(parents=True, exist_ok=True)
for filename, table_id in exports.items():
    df = client.query(f'SELECT * FROM `{table_id}`').to_dataframe()
    df = df[expected_columns[filename]]
    if 'frequencia_pct' in df:
        df['frequencia_pct'] = pd.to_numeric(df['frequencia_pct'], errors='coerce').map(lambda x: f'{x:.2f}')
    df.to_csv(RESULTADOS / filename, index=False, encoding='utf-8', na_rep='')
    print(f'Exportado: {RESULTADOS / filename} | {len(df):,} linhas')


## 9. Checklist BigQuery

- [ ] `motriz_bronze`, `motriz_silver` e `motriz_gold` existem no projeto correto.
- [ ] Bronze contém as tabelas extraídas e, se carregados, os 12 CSVs originais da Rede C.
- [ ] Silver contém `frequencia_todas_redes` com `data` como DATE, particionamento diário e clusterização por rede/escola/turma.
- [ ] Gold contém as três tabelas analíticas calculadas em SQL.
- [ ] Contagens e percentuais foram comparados com os CSVs do Notebook 01.
- [ ] Nenhuma chave ou credencial foi salva no notebook ou enviada ao GitHub.

### Observações para o README

- Bronze guarda extrações em tabelas separadas e mantém os CSVs originais da Rede C em tabelas individuais; os arquivos-fonte também ficam no Drive para rastreabilidade.
- Silver é a tabela consolidada de frequência validada; particionamento por `data` reduz leitura em consultas filtradas por período, e clusterização por `rede`, `codigo_inep` e `turma` favorece consultas frequentes.
- Gold materializa os indicadores exigidos, em vez de recalculá-los no cliente Python.
- A carga usa substituição integral das tabelas alvo (`WRITE_TRUNCATE`) e SQL `CREATE OR REPLACE`, portanto é idempotente. Para grandes volumes de produção, uma estratégia incremental com `MERGE` e controle de watermark pode ser preferível.
- BigQuery Sandbox tem limites de uso e retenção próprios; monitore o uso e não habilite recursos pagos para este desafio.
